In [2]:
import pandas_ta as pta
import pandas as pd
import json
import plotly.express as px
import chart_studio.plotly as py
import plotly.graph_objects as go
import talib.abstract as ta

In [ ]:
columns = {0:"time",1:"open",2:"high",3:"low",4:"close",5:"volume"}
dataframe = pd.read_json("BTC_USDT-1m.json").reset_index()
dataframe.rename(columns = columns, inplace = True)
dataframe = dataframe[-10080:]
values = [(10,3),(25,5)]
for l, m in values:
    dataframe[[f'supert_{l}_{m}',f'supertd_{l}_{m}',f'supertl_{l}_{m}',f'superts_{l}_{m}']] = pta.supertrend(
        dataframe.high,
        dataframe.low,
        dataframe.close,
        length=l, 
        multiplier= m
    )
    dataframe[f'supertd_{l}_{m}_previous'] = dataframe[f'supertd_{l}_{m}'].shift(1)
# dataframe['ema'] = pta.ema(dataframe['close'], 200)

dataframe = dataframe[-1440:].reset_index()
dataframe.loc[
    (
        # (dataframe['supert_25_5'] < dataframe['close']) &
        (dataframe['supertd_25_5'] == 1) &
        (dataframe['supertd_25_5_previous'] == -1) &
        # (dataframe['supertd_10_3'] == 1) &
        # (dataframe['supert_10_3'] > dataframe['supert_25_5']) &
        (dataframe['volume'] > 0)
),
    'enter_long'] = 1
dataframe.loc[
    (
        # (dataframe['supertd_25_5'] == 1) &
        (dataframe['supertd_25_5'] == -1) &
        (dataframe['supertd_25_5_previous'] == 1)
        # (dataframe['supert_25_5'] > dataframe['close']) &
        # (dataframe['supertd_25_5_previous'] != dataframe['supertd_25_5']) &
        # (dataframe['supertd_10_3_previous'] != dataframe['supertd_10_3'])
),
    'exit_long'] = 1
df_ = dataframe[['close','supert_25_5','supertd_25_5','supert_10_3', 'supertd_10_3']]
fig = px.line(x=dataframe.index,y=dataframe.close)
fig.add_scatter(x=dataframe.index,y=dataframe['supert_10_3'],line_width=1, line_color = "pink")
# fig.add_scatter(x=dataframe.index,y=dataframe['superts_10_3'],line_width=1, line_color = "red")
fig.add_scatter(x=dataframe.index,y=dataframe['supert_25_5'], line_width=1, line_color = "orange")
# fig.add_scatter(x=dataframe.index,y=dataframe.ema)
df = dataframe[dataframe['enter_long'] == 1]
fig.add_scatter(x=df.index,y=df.close, mode="markers", marker_size = 10, marker_symbol="diamond", marker_color="green")
df = dataframe[dataframe['exit_long'] == 1]
fig.add_scatter(x=df.index,y=df.close, mode="markers", marker_size = 10, marker_color="red")
go.FigureWidget(fig)


In [3]:
columns = {0:"time",1:"open",2:"high",3:"low",4:"close",5:"volume"}
dataframe = pd.read_json("BTC_USDT-1m.json").reset_index()
dataframe.rename(columns = columns, inplace = True)

In [4]:
dataframe['adx'] = ta.ADX(dataframe)
dataframe['slowadx'] = ta.ADX(dataframe, 35)

dataframe['cci'] = ta.CCI(dataframe)

stoch = ta.STOCHF(dataframe, 5)
dataframe['fastd'] = stoch['fastd']
dataframe['fastk'] = stoch['fastk']
dataframe['fastk-previous'] = dataframe.fastk.shift(1)
dataframe['fastd-previous'] = dataframe.fastd.shift(1)

slowstoch = ta.STOCHF(dataframe, 50)
dataframe['slowfastd'] = slowstoch['fastd']
dataframe['slowfastk'] = slowstoch['fastk']
dataframe['slowfastk-previous'] = dataframe.slowfastk.shift(1)
dataframe['slowfastd-previous'] = dataframe.slowfastd.shift(1)

dataframe['ema5'] = ta.EMA(dataframe, timeperiod=5)
dataframe['mean-volume'] = dataframe['volume'].rolling(12).mean()

In [5]:
dataframe.loc[
    (
        (
            (dataframe['adx'] > 50) |
            (dataframe['slowadx'] > 26)
        ) &
        (dataframe['cci'] < -100) &
        (
            (dataframe['fastk-previous'] < 20) &
            (dataframe['fastd-previous'] < 20)
        ) &
        (
            (dataframe['slowfastk-previous'] < 30) &
            (dataframe['slowfastd-previous'] < 30)
        ) &
        (dataframe['fastk-previous'] < dataframe['fastd-previous']) &
        (dataframe['fastk'] > dataframe['fastd']) &
        (dataframe['mean-volume'] > 0.75) &
        (dataframe['close'] > 0.00000100)
    ),
    'enter_long'] = 1

In [6]:
dataframe.loc[

    (
        (dataframe['slowadx'] < 25) &
        ((dataframe['fastk'] > 70) | (dataframe['fastd'] > 70)) &
        (dataframe['fastk-previous'] < dataframe['fastd-previous']) &
        (dataframe['close'] > dataframe['ema5'])
    ),
    'exit_long'] = 1

In [ ]:
fig = px.line(x=dataframe.index,y=dataframe.close)
fig.add_scatter(x=dataframe.index,y=dataframe['supert_10_3'],line_width=1, line_color = "pink")
# fig.add_scatter(x=dataframe.index,y=dataframe['superts_10_3'],line_width=1, line_color = "red")
fig.add_scatter(x=dataframe.index,y=dataframe['supert_25_5'], line_width=1, line_color = "orange")
# fig.add_scatter(x=dataframe.index,y=dataframe.ema)
df = dataframe[dataframe['enter_long'] == 1]
fig.add_scatter(x=df.index,y=df.close, mode="markers", marker_size = 10, marker_symbol="diamond", marker_color="green")
df = dataframe[dataframe['exit_long'] == 1]
fig.add_scatter(x=df.index,y=df.close, mode="markers", marker_size = 10, marker_color="red")
go.FigureWidget(fig)

In [3]:
!pip install -r r.txt

  Using cached aiodns-3.0.0-py3-none-any.whl (5.0 kB)
  Using cached aiofiles-23.1.0-py3-none-any.whl (14 kB)
  Using cached aiohttp-3.8.4-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (1.0 MB)
  Using cached aiosignal-1.3.1-py3-none-any.whl (7.6 kB)
  Using cached anyio-3.6.2-py3-none-any.whl (80 kB)
  Using cached APScheduler-3.6.3-py2.py3-none-any.whl (58 kB)
  Using cached arrow-1.2.3-py3-none-any.whl (66 kB)
  Using cached ast_comments-1.0.1-py3-none-any.whl (4.2 kB)
  Using cached async_timeout-4.0.2-py3-none-any.whl (5.8 kB)
  Using cached attrs-22.2.0-py3-none-any.whl (60 kB)
  Using cached blosc-1.11.1-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.5 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 5.0 MB/s eta 0:00:0000:0100:010m
  Using cached cachetools-4.2.2-py3-none-any.whl (11 kB)
  Using cached ccxt-3.0.50-py2.py3-none-any.whl (3.6 MB)
  Using cached certifi-2022.12.7-py3-none-any.whl (155 kB)
  Using cached cffi-1.15.1-cp39-cp39-many